# Create HAL config and shutter files

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/<variant>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs import (
    get_frame_table, get_transit_frame_table, get_color_sequence_name,
    create_shutter_file, create_hal_config, power_dict_to_channel_list,
    frame_table_filename, shutter_filename, hal_config_filename,
    copy_mosaic_helper_configs,
)
from MERci.acquisition.display import print_frame_table, display_xml
from MERci.visualization       import visualize_shutter_sequence

In [ ]:
SETTINGS_DIR = SAMPLE_DIR / "settings"
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

MICROSCOPE = "MF3"

# ── Auto-detect HAL config template ────────────────────────────────────
# Looks for hal-config-*{MICROSCOPE}*.xml in MERci/data/configs/hal/
# (case-insensitive match on the microscope name)
_hal_dir        = MERCI_DIR / "data" / "configs" / "hal"
_hal_candidates = sorted(
    p for p in _hal_dir.glob("hal-config-*.xml")
    if MICROSCOPE.lower() in p.name.lower()
)
if not _hal_candidates:
    raise FileNotFoundError(
        f"No HAL template found for microscope '{MICROSCOPE}' in {_hal_dir}"
    )
HAL_TEMPLATE = _hal_candidates[0]

# ── Imaging file format and camera settings ────────────────────────────
# FILE_TYPE options: ".zarr" (default), ".dax", ".tiff"
FILE_TYPE     = ".zarr"
EXPOSURE_TIME = 0.25      # seconds

# ── Per-channel laser power ─────────────────────────────────────────────
# POWER maps each excitation wavelength (nm) to its laser power (0..1). It is
# written ONLY into the HAL config's <default_power> list, ordered per channel
# via the microscope colour->channel map (power_dict_to_channel_list) -- this
# is the actual acquisition power. Every shutter <event>'s own <power> is
# always POWER_DEFAULT (1.0) regardless of colour: it is a full-modulation
# flag relative to <default_power>, not an independent absolute power, so
# writing the same real per-colour intensity into both places double-applies
# the scaling on real hardware (a bug this notebook had for a while -- see
# create_shutter_file's docstring).
# Any wavelength not listed falls back to POWER_DEFAULT.
POWER = {
    750: 1.00,
    650: 1.00,
    560: 1.00,
    488: 0.08,
    405: 0.05,
}
POWER_DEFAULT = 1.0

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SETTINGS_DIR : {SETTINGS_DIR}")
print(f"METADATA_DIR : {METADATA_DIR}")
print(f"HAL template : {HAL_TEMPLATE.name}")
print(f"Power (nm->power) : {POWER}")
print(f"HAL default_power : {power_dict_to_channel_list(POWER, MICROSCOPE, POWER_DEFAULT)}")

## create hal config for focus test

A short, representative movie (`FOCUS_TEST_COLOR_SEQ`, e.g. `[488, nan]`,
imaged at a single z position) used by `04_create_dave_config.ipynb`'s
optional focus-lock test recipe (`N_TEST_FRAMES > 0`) to take a real short
movie per FOV and verify focus lock quality before committing to the full
multi-hour acquisition -- generated here, alongside the other HAL configs,
using the same `EXPOSURE_TIME`/`POWER` as the real rounds below.

In [ ]:
# ── Define focus-test imaging sequence ───────────────────────────────────
z_bead       = 0
FOCUS_TEST_Z = 0.5             # single z position (um above bead_z) -- just enough
                                 # for a real per-FOV .off sidecar, not a full z-stack
color_seq    = [488, np.nan]
z_pos        = np.array([FOCUS_TEST_Z])

frame_table = get_frame_table(z_bead, bead_seq=[], color_seq=color_seq, end_seq=[], z_pos=z_pos,
                               microscope=MICROSCOPE)
name        = get_color_sequence_name(frame_table)
print(f"Color sequence name: {name}")
print_frame_table(frame_table)

# ── Save frame table ────────────────────────────────────────────────────
ft_path = METADATA_DIR / frame_table_filename("focustest", name)
frame_table.to_csv(ft_path)
print(f"Frame table saved: {ft_path}")

# ── Write shutter file (always full power -- POWER only sets HAL <default_power> below) ──
shutter_name = shutter_filename("focustest", name)
shutter_path = SETTINGS_DIR / shutter_name
create_shutter_file(frame_table, shutter_path, default_power=POWER_DEFAULT)
print(f"Shutter file saved: {shutter_path}")
display_xml(shutter_path)

# ── Write HAL config (per-channel <default_power> from POWER) ────────────
hal_name   = hal_config_filename(MICROSCOPE, "focustest", name)
hal_output = SETTINGS_DIR / hal_name
create_hal_config(HAL_TEMPLATE, frame_table, shutter_name, hal_output,
                  default_power=power_dict_to_channel_list(POWER, MICROSCOPE, POWER_DEFAULT),
                  file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME)
print(f"HAL config saved: {hal_output}")
display_xml(hal_output)

# ── Visualise shutter sequence ──────────────────────────────────────────
visualize_shutter_sequence(
    frame_table,
    title=f"Shutter sequence: {name}",
    save_path=METADATA_DIR / f"shutter_sequence_{name}.png",
)

## create hal config for cells

In [ ]:
# ── Define imaging sequence ────────────────────────────────────────────
z_bead    = 0
z_min     = 1
z_max     = 25
z_step    = 1
z_pos     = np.arange(z_min, z_max+1, z_step) 
bead_seq  = [488]
color_seq = [405]
end_seq   = [488]

# Scan mode (see bits block for description; typically "interleaved" for a single-color round)
SCAN_MODE = "interleaved"

# z return mode — how the objective travels back to z_bead after the stack:
#   "progressive" : blank frames step the objective down to z_bead in increments of RETURN_STEP
#   "instant"     : single jump back to z_bead (no intermediate frames)
Z_RETURN_MODE = "progressive"
RETURN_STEP   = 5


frame_table = get_frame_table(z_bead, bead_seq, color_seq, end_seq, z_pos,
                               microscope=MICROSCOPE, scan_mode=SCAN_MODE,
                               z_return_mode=Z_RETURN_MODE, return_step=RETURN_STEP)
name        = get_color_sequence_name(frame_table, scan_mode=SCAN_MODE)
print(f"Color sequence name: {name}")
print_frame_table(frame_table)

# ── Save frame table ────────────────────────────────────────────────────
ft_path = METADATA_DIR / frame_table_filename("cells", name)
frame_table.to_csv(ft_path)
print(f"Frame table saved: {ft_path}")

# ── Write shutter file (always full power -- POWER only sets HAL <default_power> below) ──
shutter_name = shutter_filename("cells", name)
shutter_path = SETTINGS_DIR / shutter_name
create_shutter_file(frame_table, shutter_path, default_power=POWER_DEFAULT)
print(f"Shutter file saved: {shutter_path}")
display_xml(shutter_path)

# ── Write HAL config (per-channel <default_power> from POWER) ────────────
hal_name   = hal_config_filename(MICROSCOPE, "cells", name)
hal_output = SETTINGS_DIR / hal_name
create_hal_config(HAL_TEMPLATE, frame_table, shutter_name, hal_output,
                  default_power=power_dict_to_channel_list(POWER, MICROSCOPE, POWER_DEFAULT),
                  file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME)
print(f"HAL config saved: {hal_output}")
display_xml(hal_output)

# ── Visualise shutter sequence ──────────────────────────────────────────
visualize_shutter_sequence(
    frame_table,
    title=f"Shutter sequence: {name}",
    save_path=METADATA_DIR / f"shutter_sequence_{name}.png",
)

## create hal config for bits

In [ ]:
# ── Define imaging sequence ────────────────────────────────────────────
z_bead    = 0
z_min     = 1
z_max     = 25
z_step    = 1
z_pos     = np.arange(z_min, z_max+1, z_step) 
bead_seq  = [488]
color_seq = [750,    650,    560]
end_seq   = [488]

# Scan mode — controls the order in which z-planes and colors are acquired:
#   "interleaved" : all colors acquired at each Z-nanopositioner position before stepping to the next z
#                   (AOTF / fast electronic switching)
#   "sequential"  : full z-stack acquired per color with boustrophedon Z-nanopositioner sweep,
#                   then switch to the next color (physical shutter / slow switching)
SCAN_MODE = "interleaved"

# z return mode — how the objective travels back to z_bead after the stack:
#   "progressive" : blank frames step the objective down to z_bead in increments of RETURN_STEP
#   "instant"     : single jump back to z_bead (no intermediate frames)
Z_RETURN_MODE = "progressive"
RETURN_STEP   = 5


frame_table = get_frame_table(z_bead, bead_seq, color_seq, end_seq, z_pos,
                               microscope=MICROSCOPE, scan_mode=SCAN_MODE,
                               z_return_mode=Z_RETURN_MODE, return_step=RETURN_STEP)
name        = get_color_sequence_name(frame_table, scan_mode=SCAN_MODE)
print(f"Color sequence name: {name}")
print_frame_table(frame_table)

# ── Save frame table ────────────────────────────────────────────────────
ft_path = METADATA_DIR / frame_table_filename("bits", name)
frame_table.to_csv(ft_path)
print(f"Frame table saved: {ft_path}")

# ── Write shutter file (always full power -- POWER only sets HAL <default_power> below) ──
shutter_name = shutter_filename("bits", name)
shutter_path = SETTINGS_DIR / shutter_name
create_shutter_file(frame_table, shutter_path, default_power=POWER_DEFAULT)
print(f"Shutter file saved: {shutter_path}")
display_xml(shutter_path)

# ── Write HAL config (per-channel <default_power> from POWER) ────────────
hal_name   = hal_config_filename(MICROSCOPE, "bits", name)
hal_output = SETTINGS_DIR / hal_name
create_hal_config(HAL_TEMPLATE, frame_table, shutter_name, hal_output,
                  default_power=power_dict_to_channel_list(POWER, MICROSCOPE, POWER_DEFAULT),
                  file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME)
print(f"HAL config saved: {hal_output}")
display_xml(hal_output)

# ── Visualise shutter sequence ──────────────────────────────────────────
visualize_shutter_sequence(
    frame_table,
    title=f"Shutter sequence: {name}",
    save_path=METADATA_DIR / f"shutter_sequence_{name}.png",
)

## create hal config for transit

Transit FOVs sit between two tissue boundaries and are visited only to move the
stage smoothly between sections (see notebook 02). No data is collected there:
each transit FOV is imaged as `N_TRANSIT_BLANK` blank (laser-off) frames held at
the bead focus. This produces a dedicated `hal-config-{mic}-transit-*.xml` +
shutter that notebook 03 assigns to the transit movies.

In [ ]:
# ── Define transit acquisition ─────────────────────────────────────────
z_bead          = 0     # bead focus; transit frames stay here
N_TRANSIT_BLANK = 2     # blank frames per transit FOV

transit_ft   = get_transit_frame_table(bead_z=z_bead, n_blank=N_TRANSIT_BLANK)
transit_name = get_color_sequence_name(transit_ft)   # e.g. "blkf2"
print(f"Transit sequence name: {transit_name}  ({N_TRANSIT_BLANK} blank frames)")
print_frame_table(transit_ft)

# ── Save frame table ────────────────────────────────────────────────────
ft_path = METADATA_DIR / frame_table_filename("transit", transit_name)
transit_ft.to_csv(ft_path)
print(f"Frame table saved: {ft_path}")

# ── Write shutter file (all frames blank -> no events) ───────────────────
shutter_name = shutter_filename("transit", transit_name)
shutter_path = SETTINGS_DIR / shutter_name
create_shutter_file(transit_ft, shutter_path)
print(f"Shutter file saved: {shutter_path}")
display_xml(shutter_path)

# ── Write HAL config ────────────────────────────────────────────────────
hal_name   = hal_config_filename(MICROSCOPE, "transit", transit_name)
hal_output = SETTINGS_DIR / hal_name
create_hal_config(HAL_TEMPLATE, transit_ft, shutter_name, hal_output,
                  default_power=power_dict_to_channel_list(POWER, MICROSCOPE, POWER_DEFAULT),
                  file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME)
print(f"HAL config saved: {hal_output}")
display_xml(hal_output)

## Copy mosaic-helper configs (optional)

Some microscopes have hand-crafted 10x/60x HAL+shutter configs used to set
up imaging with the Steve mosaic tool (`data/configs/hal/mosaic_helper/`,
kept in a dedicated subfolder so they never get picked up by the HAL-template
auto-detection glob above). If `MICROSCOPE` has them, they're copied into
`settings/` alongside the configs above; if not, this just prints that none
are available -- not an error, since not every microscope has these.

In [ ]:
MOSAIC_HELPER_DIR = MERCI_DIR / "data" / "configs" / "hal" / "mosaic_helper"
mosaic_helper_copied = copy_mosaic_helper_configs(MICROSCOPE, MOSAIC_HELPER_DIR, SETTINGS_DIR)
if mosaic_helper_copied:
    print(f"Copied {len(mosaic_helper_copied)} mosaic-helper config(s) for {MICROSCOPE}:")
    for p in mosaic_helper_copied:
        print(f"  {p.name}")
else:
    print(f"No mosaic-helper configs available for {MICROSCOPE} -- skipping.")

## Kilroy config

Locates the Kilroy config for `MICROSCOPE` in `MERci/data/configs/kilroy/`
(newest matching file by `YYMMDD` date stamp, MF2 fallback if `MICROSCOPE`
has none) and copies it to `SAMPLE_DIR/settings/`, alongside the HAL configs
above. `04_create_dave_config.ipynb` resolves this same file again (same
lookup, same fallback) to use as the Kilroy protocol source for the Dave
recipe -- copying it here just makes it available in `settings/` on the
acquisition computer independently of when notebook 04 is run.

In [ ]:
import shutil
from MERci.acquisition.kilroy import find_kilroy_config

KILROY_DIR  = MERCI_DIR / "data" / "configs" / "kilroy"
kilroy_src  = find_kilroy_config(MICROSCOPE, KILROY_DIR, fallback_microscope="MF2")
kilroy_dest = SETTINGS_DIR / kilroy_src.name
shutil.copy2(str(kilroy_src), str(kilroy_dest))
print(f"Kilroy config  : {kilroy_src.name}")
print(f"Copied to      : {kilroy_dest}")